In [ ]:
from dotenv import load_dotenv
from src.db.chroma_db import ChromaDb
from src.models.openai_provider import OpenAILLMProvider, OpenAIEmbeddingProvider
from src.prompts import timeframe_detection_system, timeframe_detection_user
from src.schemas.schemas import TimeframeDetection, InputLanguage
from src.services.nkod_data_processor import NkodDataProcessor
from src.db.graph_db import GraphDb
from src.db.sq_lite import SqLite
from datetime import date
from src.services.language_detector import LanguageDetector
from src.services.nkod_query_matcher import NkodQueryMatcher
from src.services.timeframe_detector import TimeframeDetector
from src.evaluators.nkod_query_matcher_evaluator import NkodQueryMatcherEvaluator
from src.services.nkod_query_matcher_reranker import NkodQueryMatcherReranker
from src.services.nkod_rag import NkodRAG


load_dotenv()

## Downloading and creating SQL tables for the NKOD metadata


In [ ]:
import pandas as pd
from src.services.nkod_data_processor import NkodDataProcessor
from src.utils import dir_name_from_uri

metadata_df = pd.read_csv(NkodDataProcessor("nkod").ofn_metadata_csv_path)
print(metadata_df.shape)
#metadata_df = metadata_df[metadata_df['dataset_uri'].apply(dir_name_from_uri) != "999788533"]
#metadata_df.to_csv(NkodDataProcessor("nkod").ofn_metadata_csv_path, index=False)

In [ ]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)

#nkod_data_processor.create_dataset_publisher_csv(graph_db)
nkod_data_processor.download_catalog_metadata()
nkod_data_processor.download_catalog_distributions()
nkod_data_processor.download_catalog_datasets()
nkod_data_processor.create_metadata_csv(graph_db)
nkod_data_processor.create_themes_csv(graph_db)
nkod_data_processor.create_metadata_sql(sq_lite)
nkod_data_processor.create_themes_sql(sq_lite)

In [ ]:
from src.pipelines.nkod_query_matcher_pipeline import NkodQueryMatcherPipeline
from src.schemas.nkod_query_matcher_request import NkodQueryMatcherRequest
from src.services.entity_generator import EntityGenerator
from src.services.nkod_data_processor import NkodDataProcessor
from src.models.openai_provider import OpenAILLMProvider, OpenAIEmbeddingProvider
from src.db.chroma_db import ChromaDb
from src.services.nkod_query_matcher import NkodQueryMatcher

nkod_data_processor = NkodDataProcessor("nkod")
openai_embeddings = OpenAIEmbeddingProvider(model_name="text-embedding-3-large", dimensions=None)
chroma_db = ChromaDb(nkod_data_processor.vectordb_path)
query = "Jaký druh stromu se v posledním roce procentuálně nejvíce obnovoval v Libereckém kraji?"
nkod_query_matcher = NkodQueryMatcher(query)
entity_generator = EntityGenerator()
#print(nkod_query_matcher.get_matching_entitities_keywords(30, chroma_db, nkod_data_processor, "cs", openai_embeddings, entity_generator))


rq = NkodQueryMatcherRequest(
    query=query,
    llm_provider="openai",
    language="cs",
    model_name="gpt-5"
)

matching_pipeline = NkodQueryMatcherPipeline().run(rq)

In [ ]:
import pandas as pd

data = [
    {"name": "Alice", "age": 30, "city": "Sydney"},
    {"name": "Bob", "age": 25, "city": "Melbourne"},
    {"name": "Charlie", "age": 35, "city": "Brisbane"}
]

df = pd.DataFrame(data)
print(df)

## Indexing the keywords, titles and descriptions from the NKOD metadata

In [ ]:
openai_embeddings = OpenAIEmbeddingProvider(model_name="text-embedding-3-large", dimensions=1536)
chroma_db = ChromaDb(nkod_data_processor.vectordb_path)

nkod_data_processor.index_catalog_themes(sq_lite, openai_embeddings, chroma_db)
nkod_data_processor.index_catalog_metadata(sq_lite, openai_embeddings, chroma_db, verbose=True)
print(chroma_db.list_collections())

## Language detection, Timeframe detection and Query matching on OFN dataset

In [ ]:
model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)
nkod_query_evaluator = NkodQueryMatcherEvaluator()
nkod_query_reranker = NkodQueryMatcherReranker()

k = 30
evaluation_dataset = "ofn_dataset_ofn_new.jsonl"
nkod_query_evaluator.evaluate_on_ofn_dataset(k, evaluation_dataset, chroma_db, nkod_data_processor, "cs", openai_embeddings, nkod_query_reranker, openai_llm)

## Language detection, Timeframe detection and Query matching on LLM dataset

In [ ]:
from rdflib import Graph
g = Graph()
g.parse("https://data.mff.cuni.cz/soubory/číselníky/organizační-struktura.ofn.jsonld", format="json-ld")
print(list(g.query("""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n
SELECT DISTINCT ?rel ?com\n
        WHERE { \n
        ?rel a/rdfs:subPropertyOf* rdf:Property . \n
        OPTIONAL { ?rel rdfs:comment ?com } \n
        }
""")))

In [ ]:
from shaclgen.shaclgen import data_graph
from rdflib import Graph

source_graph = Graph()
source_graph.parse("https://data.mff.cuni.cz/soubory/čoi/coi.trig", format="trig")

extraction_graph = data_graph(source_graph)
shacl_graph = extraction_graph.gen_graph()
print(shacl_graph)

In [ ]:
print(shacl_graph.serialize(format='trig'))
shacl_graph.print("trig")
